In [2]:
from pathlib import Path
import mlflow
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("Paso_2_MLflow_sintetico")

X, y = make_classification(
    n_samples=1200,
    n_features=12,
    n_informative=8,
    n_redundant=2,
    n_classes=2,
    weights=[0.7, 0.3],
    flip_y=0.04,
    class_sep=1.2,
    random_state=42,
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

feature_names = [f"feature_{i}" for i in range(X.shape[1])]
X_train_df = pd.DataFrame(X_train, columns=feature_names)
X_test_df = pd.DataFrame(X_test, columns=feature_names)

print("Forma de los datos:", X.shape)
print("Distribuci?n objetivo:", pd.Series(y).value_counts().to_dict())


Forma de los datos: (1200, 12)
Distribuci?n objetivo: {0: 836, 1: 364}


### 1) Modelo base

Se prueba un pipeline con escalado y regresion logistica, que suele ser una buena linea base para este tipo de clasificacion.

In [3]:
baseline_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000, C=0.5))
])

baseline_model.fit(X_train_df, y_train)
y_pred_base = baseline_model.predict(X_test_df)
y_proba_base = baseline_model.predict_proba(X_test_df)[:, 1]

baseline_metrics = {
    "accuracy": accuracy_score(y_test, y_pred_base),
    "precision": precision_score(y_test, y_pred_base, zero_division=0),
    "recall": recall_score(y_test, y_pred_base, zero_division=0),
    "f1": f1_score(y_test, y_pred_base, zero_division=0),
    "roc_auc": roc_auc_score(y_test, y_proba_base),
}

print("Base:", baseline_metrics)


Base: {'accuracy': 0.8166666666666667, 'precision': 0.7307692307692307, 'recall': 0.6263736263736264, 'f1': 0.6745562130177515, 'roc_auc': 0.8912666281087334}


### 2) Mejora del algoritmo

La clase minoritaria tiene menos peso, asi que ajustamos el hiperparametro `class_weight="balanced"` para mejorar el recall sin perder demasiado la precision.

In [4]:
improved_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000, C=2.0, class_weight="balanced"))
])

improved_model.fit(X_train_df, y_train)
y_pred_improved = improved_model.predict(X_test_df)
y_proba_improved = improved_model.predict_proba(X_test_df)[:, 1]

improved_metrics = {
    "accuracy": accuracy_score(y_test, y_pred_improved),
    "precision": precision_score(y_test, y_pred_improved, zero_division=0),
    "recall": recall_score(y_test, y_pred_improved, zero_division=0),
    "f1": f1_score(y_test, y_pred_improved, zero_division=0),
    "roc_auc": roc_auc_score(y_test, y_proba_improved),
}

print("Mejorado:", improved_metrics)
comparison = pd.DataFrame(
    [{"modelo": "base", **baseline_metrics}, {"modelo": "mejorado", **improved_metrics}]
)
display(comparison)


Mejorado: {'accuracy': 0.82, 'precision': 0.6504065040650406, 'recall': 0.8791208791208791, 'f1': 0.7476635514018691, 'roc_auc': 0.8896366791103634}


,modelo,accuracy,precision,recall,f1,roc_auc
0,base,0.816667,0.730769,0.626374,0.674556,0.891267
1,mejorado,0.820000,0.650407,0.879121,0.747664,0.889637


### 3) Registro del experimento con MLflow

Ahora registramos ambos modelos en MLflow para comparar metricas y artefactos desde la UI.

In [5]:
for run_name, model, metrics in [
    ("baseline", baseline_model, baseline_metrics),
    ("improved", improved_model, improved_metrics),
]:
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({
            "model_name": run_name,
            "solver": "lbfgs",
            "scaler": "StandardScaler",
            "C": 0.5 if run_name == "baseline" else 2.0,
            "class_weight": "balanced" if run_name == "improved" else "none",
        })
        mlflow.log_metrics(metrics)
        mlflow.sklearn.log_model(model, name="model")
        print("Run registrado:", mlflow.active_run().info.run_id)

runs = mlflow.search_runs(search_all_experiments=True)
print(runs[["tags.mlflow.runName", "metrics.accuracy", "metrics.f1", "metrics.roc_auc"]])


Run registrado: a1c1f56c3de8446b8ed7e68763996606
Run registrado: 9984440715ce42e9ab610f4e5c5ac983
  tags.mlflow.runName  metrics.accuracy  metrics.f1  metrics.roc_auc
0            improved          0.820000    0.747664         0.889637
1            baseline          0.816667    0.674556         0.891267
2            improved          0.820000    0.747664         0.889637
3            baseline          0.816667    0.674556         0.891267
4            baseline          0.816667    0.674556         0.891267


### Conclusion

La mejora principal fue ajustar la clase minoritaria con `class_weight="balanced"`, que aumento el recall y mejoro el F1 score. En proyectos reales, este tipo de experimentacion es util para comparar modelos rapidamente y conservar una trazabilidad completa con MLflow.